GitHub Project Information

maria-korosteleva/Garment-Pattern-Estimation
Stars: 181
Forks: 21
Last Commit Date: December 23, 2024

tf-encrypted/tf-encrypted
Stars: 1.2k
Forks: 211
Last Commit Date: September 25, 2024

baggepinnen/SpectralDistances.jl
Stars: 49
Forks: 5
Last Commit Date: January 15, 2026

MassimoCimmino/pygfunction
Stars: 72
Forks: 26
Last Commit Date: August 4, 2025

the-virtual-brain/tvb-root
Stars: 190
Forks: 151
Last Commit Date: September 23, 2026

Xtra-Computing/thundergbm
Stars: 715
Forks: 88
Last Commit Date: March 19, 2025

iTowns/itowns
Stars: 1.3k
Forks: 317
Last Commit Date: September 14, 2026

Serock3/Master-Thesis-QEC-simulation
Stars: 5
Forks: 0
Last Commit Date: September 7, 2024

raysect/source
Stars: 112
Forks: 25
Last Commit Date: August 25, 2025

nimble-dev/nimble
Stars: 198
Forks: 27
Last Commit Date: August 31, 2026

In [ ]:
!python -m pip install -U python-woc pandas matplotlib

In [ ]:
%pip install python-woc

In [1]:
from woc.remote import WocMapsRemote
from tqdm import tqdm
import pandas as pd
# creates the client
woc = WocMapsRemote( base_url="https://worldofcode.org/api/")
# if you got an API key
# woc = WocMapsRemote( base_url="https://worldofcode.org/api/", api_key="woc-XXXXXX-YYYYYY" )

# Now for each of the ten projects you were assigned get commits, e.g.
projects = [
    'maria-korosteleva/Garment-Pattern-Estimation',
    'tf-encrypted/tf-encrypted',
    'baggepinnen/SpectralDistances.jl',
    'MassimoCimmino/pygfunction',
    'the-virtual-brain/tvb-root',
    'Xtra-Computing/thundergbm',
    'iTowns/itowns',
    'Serock3/Master_thesis_QEC_simulation',
    'raysect/source',
    'nimble-dev/nimble'
]
list_df_commits = []
for prj in projects:
  # Convert GitHub repository names to WoC V2412 project names
  prj = prj.lower().replace('/','_',2)
  commits = woc.get_values('p2c', prj)
  df = pd.DataFrame(commits, columns=["sha1"])
  df['project'] = prj
  list_df_commits.append(df)
df = pd.concat(list_df_commits)
#perhaps save the list (if no errors) so you do not need to retrieve them again
df.to_csv('df_commits.csv')
df.head(1)

,sha1,project
0,005873aebe765beedd8bb2cda3184f8c89279d21,maria-korosteleva_garment-pattern-estimation


In [2]:
# If the number of commits its not very large, you can try to get them all at the same time,
# The max batch size is 10
import time
# let us first split df['sha1'] in chunks
chunks = [df['sha1'][x:x+10] for x in range(0, len(df), 10)]commit_data = []

for chunk in tqdm(chunks): # iterate over the commits
  # res, err = woc.show_content_many('commit',chunk.to_list())

  # commit.tch returns the same data but faster
  res, err = woc.get_values_many('commit.tch',chunk.to_list())
  res = {k: v[0] for k, v in res.items()}  # this conversion is necessary because of the internal implementation

  # to walk around the rate limit
  time.sleep(1)

  if err: # check for errors
    print('Got Errors', err)

  for commit_sha, commit in res.items():
    # flatten commit objects
    commit_data.append({
          'commit': commit_sha,
          'tree': commit[0],
          'parent': list(commit[1]),
          'author': commit[2][0],
          'author_time': int(commit[2][1]),
          'author_tz': commit[2][2],
          'committer': commit[3][0],
          'committer_time': int(commit[3][1]),
          'committer_tz': commit[3][2],
          'message': commit[4],
    })

df_commit_data = pd.DataFrame(commit_data)
df_commit_data = df_commit_data.merge(df, left_on='commit', right_on='sha1')
df_commit_data.to_csv('df_commit_data.csv', index=False)
df_commit_data.head(2)

 45%|███████████████████████████████████▍                                          | 1797/3955 [30:52<37:01,  1.03s/it]

Got Errors {'078aaf982f95af593cf547da4fb8b526ec916441': 'Key 078aaf982f95af593cf547da4fb8b526ec916441 not found in /da5_fast/All.sha1c/commit_7.tch'}


 46%|███████████████████████████████████▌                                          | 1803/3955 [30:59<37:01,  1.03s/it]

Got Errors {'0959f291d2d158b9e818ab41836beaa1cfa4670b': 'Key 0959f291d2d158b9e818ab41836beaa1cfa4670b not found in /da5_fast/All.sha1c/commit_9.tch'}


 46%|███████████████████████████████████▊                                          | 1819/3955 [31:15<36:40,  1.03s/it]

...

Got Errors {'8dcf58070ec47a2dc0f9e14ab8fcf4b06cda6041': 'Key 8dcf58070ec47a2dc0f9e14ab8fcf4b06cda6041 not found in /da5_fast/All.sha1c/commit_13.tch'}


 89%|███████████████████████████████████████████████████████████████████▉        | 3538/3955 [1:00:48<07:07,  1.03s/it]

Got Errors {'8feab8a3338576a4c0233022665d5b4803ee8633': 'Key 8feab8a3338576a4c0233022665d5b4803ee8633 not found in /da5_fast/All.sha1c/commit_15.tch'}


 90%|████████████████████████████████████████████████████████████████████▌       | 3567/3955 [1:01:23<06:40,  1.03s/it]


ReadTimeout: 

In [6]:
len(commit_data)

35596

In [4]:
import pickle

with open("commit_data_partial.pkl", "wb") as f:
    pickle.dump(commit_data, f)

print("Saved!")

Saved!


In [7]:
# If the number of commits its not very large, you can try to get them all at the same time,
# The max batch size is 10
import time

# let us first split df['sha1'] in chunks
chunks = [df['sha1'][x:x+10] for x in range(0, len(df), 10)]

for chunk in tqdm(chunks[3567:]):  # iterate over the commits

    try:
        # commit.tch returns the same data but faster
        res, err = woc.get_values_many('commit.tch', chunk.to_list())

    except Exception as e:
        print("Skipped chunk because of error:", e)
        continue

    res = {k: v[0] for k, v in res.items()}

    # to walk around the rate limit
    time.sleep(1)

    if err:
        print('Got Errors', err)

    for commit_sha, commit in res.items():
        commit_data.append({
            'commit': commit_sha,
            'tree': commit[0],
            'parent': list(commit[1]),
            'author': commit[2][0],
            'author_time': int(commit[2][1]),
            'author_tz': commit[2][2],
            'committer': commit[3][0],
            'committer_time': int(commit[3][1]),
            'committer_tz': commit[3][2],
            'message': commit[4],
        })

df_commit_data = pd.DataFrame(commit_data)
df_commit_data = df_commit_data.merge(df, left_on='commit', right_on='sha1')
df_commit_data.to_csv('df_commit_data.csv', index=False)

df_commit_data.head(2)

 17%|█████████████▉                                                                   | 67/388 [01:18<05:29,  1.03s/it]

Got Errors {'a94a8bba742f30a364af7b3cb8120de60500fa19': 'Key a94a8bba742f30a364af7b3cb8120de60500fa19 not found in /da5_fast/All.sha1c/commit_41.tch'}


 24%|███████████████████▊                                                             | 95/388 [01:46<04:59,  1.02s/it]

Got Errors {'b186c3dcce4174bbf3f214af22bdc68442831a77': 'Key b186c3dcce4174bbf3f214af22bdc68442831a77 not found in /da5_fast/All.sha1c/commit_49.tch'}


 31%|████████████████████████▉                                                       | 121/388 [02:13<04:35,  1.03s/it]

...

Got Errors {'f696a1818020060a7335165844ba0e2c2e1507bf': 'Key f696a1818020060a7335165844ba0e2c2e1507bf not found in /da5_fast/All.sha1c/commit_118.tch'}


 94%|███████████████████████████████████████████████████████████████████████████     | 364/388 [06:23<00:24,  1.03s/it]

Got Errors {'f98d0619527b26b6f32c4508b45a6590d00cb320': 'Key f98d0619527b26b6f32c4508b45a6590d00cb320 not found in /da5_fast/All.sha1c/commit_121.tch'}


100%|████████████████████████████████████████████████████████████████████████████████| 388/388 [06:48<00:00,  1.05s/it]


,commit,tree,parent,author,author_time,author_tz,committer,committer_time,committer_tz,message,sha1,project
0,005873aebe765beedd8bb2cda3184f8c89279d21,530c30c4267fe94b08e339ac5f21d089d9aa5075,[f73fa70c449a304bc828303df2f81747ffea5b7e],Maria Korosteleva <mariako@kaist.ac.kr>,1617698599,+0900,Maria Korosteleva <mariako@kaist.ac.kr>,1617698599,+0900,recording stats fix\n,005873aebe765beedd8bb2cda3184f8c89279d21,maria-korosteleva_garment-pattern-estimation
1,00e5150fb1212c26300fffb603049b2d3e3c2c96,2d3bceacdf6bcfb7be781c755cec2239adc2c5a5,[e50ae07b89dc364933cc38dac534cb3cb543d47b],Maria Korosteleva <mariako@kaist.ac.kr>,1590999070,+0900,Maria Korosteleva <mariako@kaist.ac.kr>,1590999070,+0900,skirt upgrade v0.2\n,00e5150fb1212c26300fffb603049b2d3e3c2c96,maria-korosteleva_garment-pattern-estimation


In [9]:
import pickle

with open("commit_data_complete.pkl", "wb") as f:
    pickle.dump(commit_data, f)

In [10]:
project_stats = df_commit_data.groupby('project').agg(
    number_of_commits=('commit', 'nunique'),
    number_of_authors=('author', 'nunique'),
    max_time=('author_time', 'max'),
    min_time=('author_time', 'min')
)

project_stats

,number_of_commits,number_of_authors,max_time,min_time
project,,,,
baggepinnen_spectraldistances.jl,759,7,1776126985,1568771532
itowns_itowns,9190,142,1762275571,1443101686
maria-korosteleva_garment-pattern-estimation,1378,11,1736211317,1582260169
massimocimmino_pygfunction,1385,22,1771443588,1502741635
nimble-dev_nimble,9730,73,1762381653,1402585122
raysect_source,2366,33,1758534567,1376432727
serock3_master_thesis_qec_simulation,496,6,1639215098,1611075452
tf-encrypted_tf-encrypted,3165,49,1727242358,1521656591
the-virtual-brain_tvb-root,10306,136,1776180337,1401877909


In [11]:
summary_df = df_commit_data[
    ['project', 'commit', 'author', 'author_time', 'message']
].copy()

summary_df.columns = [
    'project_wocid',
    'commit_sha1',
    'author',
    'time',
    'commit message'
]

summary_df.to_csv(
    'cboddick_project_summary.csv',
    sep=';',
    index=False
)

summary_df.head()

,project_wocid,commit_sha1,author,time,commit message
0,maria-korosteleva_garment-pattern-estimation,005873aebe765beedd8bb2cda3184f8c89279d21,Maria Korosteleva <mariako@kaist.ac.kr>,1617698599,recording stats fix\n
1,maria-korosteleva_garment-pattern-estimation,00e5150fb1212c26300fffb603049b2d3e3c2c96,Maria Korosteleva <mariako@kaist.ac.kr>,1590999070,skirt upgrade v0.2\n
2,maria-korosteleva_garment-pattern-estimation,013606d395963877b82b041e0b44afc36b4ec05e,mariako <mariako@kaist.ac.kr>,1621824387,transfer segmentation labels during scan imita...
3,maria-korosteleva_garment-pattern-estimation,01384216aa713d0051f13e2db0f82a1a8b23a89c,Maria Korosteleva <mariako@kaist.ac.kr>,1593576105,add feature sizes info to config\n
4,maria-korosteleva_garment-pattern-estimation,01567bf034a1e90c3525f395e6f8a9084b331035,Maria Korosteleva <mariako@kaist.ac.kr>,1606703775,"Revert ""eval stitch sides loss right away""\n\n..."
